In [2]:
import pandas as pd
import altair as alt

url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv"

df = pd.read_csv(url)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


Shape: (4747, 29)
Columns: ['observed', 'location_details', 'county', 'state', 'season', 'title', 'latitude', 'longitude', 'date', 'number', 'classification', 'geohash', 'temperature_high', 'temperature_mid', 'temperature_low', 'dew_point', 'humidity', 'cloud_cover', 'moon_phase', 'precip_intensity', 'precip_probability', 'precip_type', 'pressure', 'summary', 'uv_index', 'visibility', 'wind_bearing', 'wind_speed', 'location']


,observed,location_details,county,state,season,title,latitude,longitude,date,number,...,precip_intensity,precip_probability,precip_type,pressure,summary,uv_index,visibility,wind_bearing,wind_speed,location
0,Ed L. was salmon fishing with a companion in P...,East side of Prince William Sound,Valdez-Chitina-Whittier County,Alaska,Fall,NaN,NaN,NaN,NaN,1261.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heh i kinda feel a little dumb that im reporti...,"the road is off us rt 80, i dont know the exit...",Warren County,New Jersey,Fall,NaN,NaN,NaN,NaN,438.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I was on my way to Claremont from Lebanon on R...,Close to Claremont down 120 not far from Kings...,Sullivan County,New Hampshire,Summer,Report 55269: Dawn sighting at Stevens Brook o...,43.41549,-72.33093,2016-06-07,55269.0,...,0.001,0.7,rain,998.87,Mostly cloudy throughout the day.,6.0,9.70,262.0,0.49,POINT(-72.33093000000001 43.415490000000005)
3,I was northeast of Macy Nebraska along the Mis...,Latitude & Longitude : 42.158230 -96.344197,Thurston County,Nebraska,Spring,Report 59757: Possible daylight sighting of a ...,42.15685,-96.34203,2018-05-25,59757.0,...,0.000,0.0,NaN,1008.07,Partly cloudy in the morning.,10.0,8.25,193.0,3.33,POINT(-96.34203000000001 42.15685)
4,"While this incident occurred a long time ago, ...","Ward County, Just outside of a the Minuteman T...",Ward County,North Dakota,Spring,Report 751: Hunter describes described being s...,48.25422,-101.31660,2000-04-21,751.0,...,NaN,NaN,rain,1011.47,Partly cloudy until evening.,6.0,10.00,237.0,11.14,POINT(-101.3166 48.254220000000004)


In [3]:

# Count reports by state
state_counts = (
    df["state"]
    .dropna()
    .value_counts()
    .head(15)
    .reset_index()
)

state_counts.columns = ["State", "Reports"]

chart1 = alt.Chart(state_counts).mark_bar().encode(
    x=alt.X("Reports:Q", title="Number of Reports"),
    y=alt.Y("State:N", sort="-x", title="State"),
    tooltip=["State:N", "Reports:Q"]
).properties(
    width=650,
    height=420,
    title="Top 15 States by Bigfoot Sightings Reports"
)

chart1


alt.Chart(...)

In [4]:

# Prepare yearly report counts
df2 = df[["date", "classification"]].copy()

df2["date"] = pd.to_datetime(
    df2["date"], errors="coerce", format="mixed"
)

df2 = df2.dropna(subset=["date", "classification"])
df2["Year"] = df2["date"].dt.year

yearly = (
    df2.groupby(["Year", "classification"])
    .size()
    .reset_index(name="Reports")
)

# Create classification dropdown
classes = sorted(yearly["classification"].unique().tolist())

class_selector = alt.param(
    name="selected_class",
    value="All",
    bind=alt.binding_select(
        options=["All"] + classes,
        name="Classification: "
    )
)

# Interactive line chart
chart2 = (
    alt.Chart(yearly)
    .mark_line(point=True)
    .encode(
        x=alt.X("Year:Q", axis=alt.Axis(format="d")),
        y=alt.Y("Reports:Q", title="Number of Reports"),
        color=alt.Color("classification:N", title="Report Class"),
        tooltip=["Year:Q", "classification:N", "Reports:Q"]
    )
    .transform_filter(
        'selected_class == "All" || datum.classification == selected_class'
    )
    .add_params(class_selector)
    .properties(
        width=700,
        height=350,
        title="Bigfoot Sightings Reports Over Time"
    )
)

print("Reports with usable dates:", len(df2))
chart2


Reports with usable dates: 3797


alt.Chart(...)

In [5]:
from pathlib import Path

output = Path("bigfoot_charts")
output.mkdir(exist_ok=True)

chart1.save(output / "bigfoot_states.json")
chart2.save(output / "bigfoot_timeline.json")

print("Both charts exported successfully!")

Both charts exported successfully!


In [1]:
from pathlib import Path

print("Current folder:", Path.cwd())
print("Exported files:")
for file in Path("bigfoot_charts").glob("*.json"):
    print(file.resolve())

Current folder: /Users/qi/Downloads/IS430 python/pythonProject
Exported files:
/Users/qi/Downloads/IS430 python/pythonProject/bigfoot_charts/bigfoot_timeline.json
/Users/qi/Downloads/IS430 python/pythonProject/bigfoot_charts/bigfoot_states.json
